In [40]:
import dagshub
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# свързване с dagshub
dagshub.init(repo_owner='cvetygeorg', repo_name='my-repo', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/cvetygeorg/my-repo.mlflow")
# Създаване/избор на експеримент
mlflow.set_experiment("students-online-DT")

# Данни
df = pd.read_csv('students-online.csv')

# преобразуване на категорийните атрибути в числови
cat_cols = ['Gender', 'Degree', 'PlatformUsage', 'PreferredFormat']
ct = ColumnTransformer(transformers=[('ohe', OneHotEncoder(sparse_output=False), cat_cols)], remainder='passthrough')
data_transformed = ct.fit_transform(df)

# Вземаме новите имена на колоните (OHE колони + останалите). Превръщаме обратно в DataFrame 
new_column_names = ct.get_feature_names_out()
final_df = pd.DataFrame(data_transformed, columns=new_column_names)

X = final_df.drop(['remainder__Satisfaction', 'remainder__ID'], axis=1)
y = final_df['remainder__Satisfaction']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Параметри на модела
max_depth = 10
criterion = "entropy" # gini

with mlflow.start_run(run_name="experiment_run_10_entropy"):
    # Създаване и обучение на модел
    model = DecisionTreeClassifier(max_depth=max_depth, criterion=criterion, random_state=42)
    model.fit(X_train, y_train)
    # Прогнози
    y_pred = model.predict(X_test)
    # Мерки
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    # Логване на параметри
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("criterion", criterion)
    # Логване на мерки
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_macro", f1)
    # Запис на модела 
    mlflow.sklearn.log_model(model, name = "DT_model")

    print("Run completed")
    print("accuracy =", accuracy)
    print("f1_macro =", f1)

Initialized MLflow to track repo "cvetygeorg/my-repo"

Repository cvetygeorg/my-repo initialized!

2026/04/10 20:46:28 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run completed
accuracy = 0.75
f1_macro = 0.6666666666666666
🏃 View run experiment_run_10_entropy at: https://dagshub.com/cvetygeorg/my-repo.mlflow/#/experiments/1/runs/da77280584a34437966d9dd7e9d65bf0
🧪 View experiment at: https://dagshub.com/cvetygeorg/my-repo.mlflow/#/experiments/1
